In [56]:
import pandas as pd
import geopandas as gpd
import altair as alt
import json
from shapely.geometry import shape
from shapely import wkt
from shapely.ops import linemerge, unary_union
alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

https://developer.tmb.cat/data

# Neighbourhoods

In [ ]:
neigh = pd.read_csv("Data/Neigh/neigh.csv")

neigh["geometry"] = neigh["geom"].apply(wkt.loads)

neigh = gpd.GeoDataFrame(
    neigh,
    geometry="geometry",
    crs="EPSG:4326"
)
neigh = neigh[["name", "geometry"]]
neigh["geometry"] = neigh.buffer(0)



# Metro Stops

In [ ]:
with open('Data/Metro/metro_stops.json') as f:
    data = json.load(f)

2.17623558, 41.3760069

In [ ]:
stops = ['Catalunya','Liceu','Drassanes','Paral·lel','Poble Sec','Tarragona','Espanya','Plaça del Centre','Sants Estació','Les Corts','Maria Cristina','Palau Reial']

In [ ]:
df = []
for stop in data['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    
    if nom_estacio in stops:
        df.append({
            'name': nom_estacio,
            'stop_type': 'Metro',
            'geometry': shape(stop['geometry'])
        })



In [ ]:
point1 = shape({
    'type': 'Point',
    'coordinates': (2.167342560951652, 41.38704037682107)
})

point2 = shape({
    'type': 'Point',
    'coordinates': (2.1768302000749906, 41.39610145325573)
})

point3 = shape({
    'type': 'Point',
    'coordinates': (2.1130282510923517, 41.38913433225749)
})

point4 = shape({
    'type': 'Point',
    'coordinates': (2.114055470466503, 41.38275273204058)
})

point5 = shape({
    'type': 'Point',
    'coordinates': (2.113665627357394, 41.384072542881135)
})

point6 = shape({
    'type': 'Point',
    'coordinates': (2.1156528732436075, 41.38484302209972)
})


point7 = shape({
    'type': 'Point',
    'coordinates': (2.114321704292043, 41.38591311049023)
})

point8 = shape({
    'type': 'Point',
    'coordinates': (2.1206922984531587, 41.38747540788272)
})






df.append({
    'name': 'Ronda Universitat',
    'stop_type': 'Bus',
    'geometry': point1
})

df.append({
    'name': 'Plaça Tetuan',
    'stop_type': 'Bus',
    'geometry': point2
})


df.append({
    'name': 'Fib',
    'stop_type': 'University',
    'geometry': point3
})

df.append({
    'name': 'Belles Arts',
    'stop_type': 'University',
    'geometry': point4
})

df.append({
    'name': 'Arquitectura',
    'stop_type': 'University',
    'geometry': point5
})

df.append({
    'name': 'FME',
    'stop_type': 'University',
    'geometry': point6
})

df.append({
    'name': 'Economia i Empresa',
    'stop_type': 'University',
    'geometry': point7
})

df.append({
    'name': 'Dret',
    'stop_type': 'University',
    'geometry': point8
})


geo_df_stops = gpd.GeoDataFrame(df)


# Metro Trajectories

In [ ]:
with open('Data/Metro/metro_trajectories.json') as f:
    trajectories = json.load(f)

In [ ]:
trajectories

In [ ]:
df_trajectories = []
for trajectory in trajectories['features']:
    properties = trajectory.get('properties', {})
    nom_estacio_fi = properties.get('NOM_ESTACIO_FI')
    nom_linia = properties.get('NOM_LINIA')
    
    if nom_estacio_fi in stops and nom_linia == 'L3':
        df_trajectories.append({
            'name': nom_estacio_fi,
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories

In [ ]:
df_trajectories = pd.DataFrame(df_trajectories)
geo_df_trajectories = gpd.GeoDataFrame(df_trajectories, geometry='geometry')


In [ ]:
geo_df_stops

In [ ]:
neigh_layer = alt.Chart(neigh).mark_geoshape(
    stroke="black",
    fill="lightblue"
).properties(
    width=600,
    height=600
    
).encode(tooltip=['name'])

trajectory_layer = alt.Chart(geo_df_trajectories).mark_geoshape(
    stroke="green",
    filled = False,
    strokeWidth=3).encode(tooltip=['name']
).properties(
    width=600,
    height=600
)

point_layer = alt.Chart(geo_df_stops).mark_geoshape(
    stroke="black",size=0).encode(tooltip=['name'],color = alt.Color('stop_type:N', scale=alt.Scale(domain=['Metro', 'Bus', 'University'], range=['green', 'blue', 'orange'])))
neigh_layer + trajectory_layer + point_layer

# All Stops

In [ ]:
df_stops_all = []
for stop in data['features']:
    properties = stop.get('properties', {})
    nom_estacio = properties.get('NOM_ESTACIO')
    linia = properties.get('PICTO')
    df_stops_all.append({
            'name': nom_estacio,
            'linia': linia,
            'stop_type': 'Metro',
            'geometry': shape(stop['geometry'])
        })

df_stops_all.append({
    'name': 'Ronda Universitat',
    'stop_type': 'Bus',
    'geometry': point1
})

df_stops_all.append({
    'name': 'Plaça Tetuan',
    'stop_type': 'Bus',
    'geometry': point2
})

geo_df_stops_all = gpd.GeoDataFrame(df_stops_all, geometry='geometry')
excluded = ['FM','L10N','L10S','L11','L9N','L9S','L9NL10N','L9SL10S','TM']
geo_df_stops_all = geo_df_stops_all[~geo_df_stops_all['linia'].isin(excluded)]
geo_df_stops_all

# All trajectories

In [ ]:
df_trajectories_all = []
for trajectory in trajectories['features']:
    properties = trajectory.get('properties', {})
    tram = properties.get('NOM_TRAM_LINIA')
    nom_linia = properties.get('NOM_LINIA')
    df_trajectories_all.append({
            'tram': tram,
            'linia': nom_linia,
            'geometry': shape(trajectory['geometry'])
        })
df_trajectories_all = pd.DataFrame(df_trajectories_all)
geo_df_trajectories_all = gpd.GeoDataFrame(df_trajectories_all, geometry='geometry')
geo_df_trajectories_all = geo_df_trajectories_all[~geo_df_trajectories_all['linia'].isin(excluded)]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Inici') == False]
geo_df_trajectories_all = geo_df_trajectories_all[geo_df_trajectories_all['tram'].str.contains('Final') == False]
geo_df_trajectories_all

In [ ]:
sol1 = shape({
    'type': 'Point',
    'coordinates':  (2.1319539025975986, 41.389541783812525)
})

sol2 = shape({
    'type': 'Point',
    'coordinates':  (2.1991779497207844, 41.404303976357355)
})

sol_df = pd.DataFrame({
    'name': ['Sol','Sol'],
    'stop_type': ['Bus', 'Bus'],
    'geometry': [sol1, sol2]
})
sol_df = gpd.GeoDataFrame(sol_df, geometry='geometry')

In [ ]:
sol_df_buffer = sol_df.copy()
sol_df_buffer["geometry"] = sol_df_buffer.buffer(0.007)
sol_df_buffer

In [ ]:
geo_df_trajectories_all

In [ ]:
geo_df_stops_all

In [ ]:
trajectory_layer = alt.Chart(geo_df_trajectories_all[geo_df_trajectories_all['linia'] == 'L4']).mark_geoshape(
    strokeWidth=3,filled = False).encode(
        color=alt.Color('linia:N',scale = alt.Scale(domain=['L1', 'L2', 'L3','L4','L5'], range=['red', 'purple', 'green','yellow','blue']))).properties(
    width=600,
    height=600
)

point_layer = alt.Chart(geo_df_stops_all[geo_df_stops_all['linia'] == 'L4']).mark_geoshape(
    stroke="black").encode(tooltip=['name'],color = alt.Color('stop_type:N', scale=alt.Scale(domain=['Metro', 'Bus'], range=['green', 'blue'])))

sol_point_layer = alt.Chart(sol_df).mark_geoshape(
    stroke="black",size=0,color='black')

buffer_layer = alt.Chart(sol_df_buffer).mark_geoshape(
    stroke="black",color='lightgrey',fillOpacity=0
)



(neigh_layer + trajectory_layer + point_layer).resolve_scale(color='independent')

# BUS

In [ ]:
with open('data/bus_stops.json') as f:
    data = json.load(f)
df_bus = []
for stop in data['features']:
    properties = stop.get('properties', {})
    codi_parada = properties.get('CODI_PARADA')
    nom_parada = properties.get('NOM_PARADA')
    
    df_bus.append({
            'name': nom_parada,
            'codi': codi_parada,
            'geometry': shape(stop['geometry'])
        })
geo_df_bus = gpd.GeoDataFrame(df_bus,crs="EPSG:4326")


In [ ]:
geo_df_bus['name'].nunique()

In [ ]:
bus_layer  = alt.Chart(geo_df_bus).mark_geoshape(
    stroke="black",size=0).encode(tooltip=['name:N', 'codi:N'])

neigh_layer + bus_layer

In [ ]:
with open('data/bus_trajectories.json') as f:
    data = json.load(f)
df_bus_traj = []
for stop in data['features']:
    properties = stop.get('properties', {})
    id_recorregut = properties.get('ID_RECORREGUT')
    nom_linia = properties.get('NOM_LINIA')
    longitud = properties.get('LONGITUD')
    sentit = properties.get('DESC_SENTIT')
    paquet = properties.get('TIPUS_PAQUET')
    
    df_bus_traj.append({
            'nom_linia': nom_linia,
            'longitud': longitud,
            'id_recorregut': id_recorregut,
            'sentit': sentit,
            'paquet': paquet,
            'geometry': shape(stop['geometry'])
        })
geo_df_bus_traj = gpd.GeoDataFrame(df_bus_traj,crs="EPSG:4326")
geo_df_bus_traj = geo_df_bus_traj[geo_df_bus_traj['paquet'] == '1']


In [ ]:
geo_df_bus_traj[geo_df_bus_traj['nom_linia'] == '7']

In [ ]:

# merge the two directions of each line into one geometry and sum the lengths
geo_df_bus_traj = (geo_df_bus_traj.groupby('nom_linia', as_index=False).agg(
    geometry=('geometry', lambda s: linemerge(unary_union(list(s))))))
geo_df_bus_traj = gpd.GeoDataFrame(geo_df_bus_traj, geometry='geometry', crs='EPSG:4326')
geo_df_bus_traj.to_crs('EPSG:25831', inplace=True)
geo_df_bus_traj['length'] = geo_df_bus_traj['geometry'].length
geo_df_bus_traj.to_crs('EPSG:4326', inplace=True)
geo_df_bus_traj[geo_df_bus_traj['nom_linia'] == '7']

es tots els recorreguts duna linia, no parada a parada

hi ha més d'un recorregut per linia

In [ ]:
bus_traj_layer1  = alt.Chart(geo_df_bus_traj).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['nom_linia:N', 'longitud:N'])


alt.vconcat(neigh_layer  + bus_traj_layer1)

In [ ]:
geo_df_bus[geo_df_bus['codi'] == 64]

In [ ]:
parades_tmb = set(geo_df_bus['codi'].unique())
len(parades_tmb)

In [ ]:
bus_stops_w_line = pd.read_excel("Data/bus_stops_w_lines.xlsx")
parades_amb = set(bus_stops_w_line['Código de parada / Codi de parada'].unique())
len(parades_amb)

In [ ]:
bus_stops_w_line

In [ ]:
print(len(parades_tmb - parades_amb))
print(len(parades_amb - parades_tmb))

In [ ]:
parades_amb - parades_tmb

In [ ]:
parades_tmb - parades_amb

# FCG

In [73]:
with open('data/FGC/fcg_stops.json') as f:
    data = json.load(f)
gtfs_stops = []
for stop in data['features']:
    properties = stop.get('properties', {})
    stop_id = properties.get('stop_id')
    stop_name = properties.get('stop_name')

    
    gtfs_stops.append({
        'stop_id': stop_id,
        'stop_name': stop_name,
        'geometry': shape(stop['geometry'])

    })  
geo_df_gtfs_stops = gpd.GeoDataFrame(gtfs_stops, crs="EPSG:4326")
geo_df_gtfs_stops = geo_df_gtfs_stops[geo_df_gtfs_stops['geometry'].within(neigh['geometry'].unary_union)]
geo_df_gtfs_stops

/var/folders/95/s4thp5290fd41pw2cj8713k80000gn/T/ipykernel_41101/3238638464.py:17: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geo_df_gtfs_stops = geo_df_gtfs_stops[geo_df_gtfs_stops['geometry'].within(neigh['geometry'].unary_union)]


,stop_id,stop_name,geometry
0,PC2,Barcelona - Plaça Catalunya,POINT (2.16872 41.38563)
1,PR1,Provença,POINT (2.15803 41.39281)
2,SG,Sant Gervasi,POINT (2.14713 41.40107)
3,MN,Muntaner,POINT (2.14235 41.39853)
4,BN,La Bonanova,POINT (2.13643 41.39782)
...,...,...,...
217,VL,Baixador de Vallvidrera,POINT (2.09694 41.42009)
218,LP,Les Planes,POINT (2.09162 41.4274)
219,LP1,Les Planes,POINT (2.09162 41.4274)
220,LP2,Les Planes,POINT (2.09162 41.4274)


In [74]:
gdfs_stops_layer = alt.Chart(geo_df_gtfs_stops).mark_geoshape(
    stroke="black",size=0).encode(tooltip=['stop_id:N', 'stop_name:N'])

neigh_layer + gdfs_stops_layer

alt.LayerChart(...)

In [76]:
with open('data/FGC/fcg_trajectories.json') as f:
    data = json.load(f)
fgc_traj = []
for stop in data['features']:
    properties = stop.get('properties', {})
    route_id = properties.get('route_id')
    route_name = properties.get('route_long_name')

    
    fgc_traj.append({
        'route_id': route_id,
        'route_name': route_name,
        'geometry': shape(stop['geometry'])

    })  
geo_df_fgc_traj = gpd.GeoDataFrame(fgc_traj, crs="EPSG:4326")
invalid = ['RL2','RL1','FV','R5','R6','MM']
geo_df_fgc_traj = geo_df_fgc_traj[geo_df_fgc_traj['route_id'].isin(invalid) == False]
geo_df_fgc_traj


,route_id,route_name,geometry
0,L6,Barcelona Pl. Catalunya - Sarrià,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
1,S2,Barcelona Pl. Catalunya - Sabadell Parc del Nord,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
2,S3,Barcelona Pl. Espanya - Can Ros,"MULTILINESTRING ((2.14847 41.37452, 2.14466 41..."
3,S4,Barcelona Pl. Espanya - Olesa de Montserrat,"MULTILINESTRING ((2.14847 41.37452, 2.14466 41..."
4,S8,Barcelona Pl. Espanya - Martorell Enllaç,"MULTILINESTRING ((2.14847 41.37452, 2.14466 41..."
5,R50,Barcelona Pl. Espanya- Manresa,"MULTILINESTRING ((1.82797 41.73144, 1.82875 41..."
6,S9,Barcelona Pl. Espanya - Quatre Camins,"MULTILINESTRING ((2.14847 41.37452, 2.14466 41..."
8,R53,Barcelona Pl. Espanya - Manresa,"MULTILINESTRING ((1.82746 41.73151, 1.83018 41..."
9,L7,Barcelona Pl. Catalunya - Av Tibidabo,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."
12,S1,Barcelona Pl. Catalunya - Terrassa Nacions Unides,"MULTILINESTRING ((2.17006 41.38575, 2.16973 41..."


iguals = S3,S4,S8,R50,S9,R53,R5,R6,R60,R63,L8
iguals = S1,S2

In [ ]:
valid = ['L6','S1','S3','L7','L12']

In [80]:
all = alt.Chart(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'L8']).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['route_id:N', 'route_name:N'])

neigh_layer + all

alt.LayerChart(...)

In [ ]:
l6 = alt.Chart(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'S1']).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['route_id:N', 'route_name:N'])

l7 = alt.Chart(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'S2']).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['route_id:N', 'route_name:N'])

l8 = alt.Chart(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'L8']).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['route_id:N', 'route_name:N'])

l12 = alt.Chart(geo_df_fgc_traj[geo_df_fgc_traj['route_id'] == 'L12']).mark_geoshape(
    stroke="black",filled = False,strokeWidth=3).encode(tooltip=['route_id:N', 'route_name:N'])

alt.vconcat(neigh_layer + gdfs_stops_layer  + l6 | neigh_layer +gdfs_stops_layer  + l7 , neigh_layer + gdfs_stops_layer + l8 | neigh_layer + gdfs_stops_layer  + l12)

In [ ]:
stops = pd.read_csv("Data/gtfs_fgc/stops.txt")
stops

# Tram

In [ ]:
tram_stops_points = pd.read_excel('Data/Tram/tram_stops.xlsx')
tram_stops = tram_stops_points[['Name', 'Latitude', 'Longitude']]
tram_stops_gdf = gpd.GeoDataFrame(tram_stops, geometry=gpd.points_from_xy(tram_stops.Longitude, tram_stops.Latitude), crs="EPSG:4326")
tram_stops_gdf



In [30]:
tram_stops_layer = alt.Chart(tram_stops_gdf).mark_geoshape(
    stroke="black",size=0).encode(tooltip=['Name:N'])

neigh_layer + tram_stops_layer

alt.LayerChart(...)

route_id =  2 -> T2 ....

In [62]:
tram_ids = pd.read_csv("Data/Tram/TBX/trips.txt")
tram_ids.groupby(['route_id','shape_id']).size().reset_index(name='counts').sort_values(by='counts', ascending=False)

,route_id,shape_id,counts
10,3,11,188
9,3,5,179
5,2,2,171
7,2,10,151
4,2,1,150
6,2,3,130
11,3,12,109
8,3,4,101
0,1,6,100
3,1,9,89


In [63]:
t1_shapes = tram_ids[tram_ids['route_id'] == 1]['shape_id'].unique().tolist()
t2_shapes = tram_ids[tram_ids['route_id'] == 2]['shape_id'].unique().tolist()
t3_shapes = tram_ids[tram_ids['route_id'] == 3]['shape_id'].unique().tolist()

In [64]:
tram_traj = pd.read_csv("Data/Tram/TBX/shapes.txt")
tra_traj_gpd = gpd.GeoDataFrame(tram_traj, geometry=gpd.points_from_xy(tram_traj.shape_pt_lon, tram_traj.shape_pt_lat), crs="EPSG:4326")
tra_traj_gpd.drop(columns=['shape_pt_lat', 'shape_pt_lon','shape_pt_sequence','shape_dist_traveled'], inplace=True)

In [60]:
tra_traj_gpd

,shape_id,geometry
0,5,POINT (2.05336 41.37936)
1,5,POINT (2.05338 41.37937)
2,5,POINT (2.05362 41.37942)
3,5,POINT (2.05374 41.37945)
4,5,POINT (2.05391 41.37949)
...,...,...
9527,3,POINT (2.14204 41.39188)
9528,3,POINT (2.14227 41.39193)
9529,3,POINT (2.14247 41.39198)
9530,3,POINT (2.14289 41.39209)


In [66]:
tra_traj_gpd[tra_traj_gpd['shape_id'].isin(t1_shapes)]

,shape_id,geometry
4186,6,POINT (2.14314 41.39222)
4187,6,POINT (2.14256 41.39207)
4188,6,POINT (2.14237 41.39202)
4189,6,POINT (2.14232 41.39201)
4190,6,POINT (2.1422 41.39197)
...,...,...
7991,7,POINT (2.14204 41.39188)
7992,7,POINT (2.14227 41.39193)
7993,7,POINT (2.14247 41.39198)
7994,7,POINT (2.14289 41.39209)


In [72]:
t1 = alt.Chart(tra_traj_gpd[tra_traj_gpd['shape_id'].isin(t1_shapes)]).mark_geoshape(
    stroke="black",size=0)

t2 = alt.Chart(tra_traj_gpd[tra_traj_gpd['shape_id'].isin(t2_shapes)]).mark_geoshape(
    stroke="black",size=0)
t3 = alt.Chart(tra_traj_gpd[tra_traj_gpd['shape_id'].isin(t3_shapes)]).mark_geoshape(
    stroke="black",size=0)



neigh_layer + t1 | neigh_layer + t2 | neigh_layer + t3

alt.HConcatChart(...)